In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
from tqdm import tqdm

In [3]:
#format v2 data as v3

In [4]:
v2root="../popocatepetl-data-v2/"

In [5]:
datav2df=pd.read_feather(v2root+"data.feather")

In [6]:
datav2df["labels"]=datav2df["labels"].str.lower()

In [7]:
datav2df["image"]=v2root + datav2df["image"]

In [8]:
datav2df.columns=["path","label"]

In [9]:
datav2df["id"]=np.full(datav2df.shape[0], -1)
datav2df["video"]=datav2df["path"].str.replace(v2root+"images/","").replace(".jpg","")

In [10]:
datav2df

,path,label,id,video
0,../popocatepetl-data-v2/images/1999/ene/p01319...,fumarola,-1,1999/ene/p0131991.jpg
1,../popocatepetl-data-v2/images/1999/ene/p01319...,fumarola,-1,1999/ene/p0131992.jpg
2,../popocatepetl-data-v2/images/1999/ene/p01319...,fumarola,-1,1999/ene/p0131993.jpg
3,../popocatepetl-data-v2/images/1999/ene/p01319...,fumarola,-1,1999/ene/p0131994.jpg
4,../popocatepetl-data-v2/images/1999/ene/p01319...,fumarola,-1,1999/ene/p0131995.jpg
...,...,...,...,...
16332,../popocatepetl-data-v2/images/2024/image_94.jpg,erupcion,-1,2024/image_94.jpg
16333,../popocatepetl-data-v2/images/2024/image_95.jpg,fumarola,-1,2024/image_95.jpg
16334,../popocatepetl-data-v2/images/2024/image_97.jpg,fumarola,-1,2024/image_97.jpg
16335,../popocatepetl-data-v2/images/2024/image_98.jpg,fumarola,-1,2024/image_98.jpg


In [11]:
image_root = '../popocatepetl-data-v3/'
data=[]
for (dirpath, dirnames, filenames) in tqdm(os.walk(image_root)):
    for file in filenames:
        if file.endswith(".jpg"):
            vpath, label=os.path.split(dirpath)
            _, vid=os.path.split(vpath)
            id_, _=os.path.splitext(file)
            data.append(pd.Series({
                "path":os.path.join(dirpath, file).replace("\\","/"),
                "label":label,
                "video":vid,
                "id":id_,
            }))
data=pd.DataFrame(data)

181it [00:24,  7.44it/s]


In [12]:
data["id"]=data["id"].astype(int)

In [13]:
data

,path,label,video,id
0,../popocatepetl-data-v3/v1/fumarola/000001.jpg,fumarola,v1,1
1,../popocatepetl-data-v3/v1/fumarola/000002.jpg,fumarola,v1,2
2,../popocatepetl-data-v3/v1/fumarola/000003.jpg,fumarola,v1,3
3,../popocatepetl-data-v3/v1/fumarola/000004.jpg,fumarola,v1,4
4,../popocatepetl-data-v3/v1/fumarola/000005.jpg,fumarola,v1,5
...,...,...,...,...
224090,../popocatepetl-data-v3/v9/erupcion/000018.jpg,erupcion,v9,18
224091,../popocatepetl-data-v3/v9/erupcion/000019.jpg,erupcion,v9,19
224092,../popocatepetl-data-v3/v9/erupcion/000020.jpg,erupcion,v9,20
224093,../popocatepetl-data-v3/v9/fumarola/000001.jpg,fumarola,v9,1


In [14]:
data["label"]=data["label"].replace({"fumarola_noche":"fumarola", "erupcion_noche":"erupcion", "inactivo_noche":"inactivo",})

In [15]:
data_merged=pd.concat([datav2df,data])
data=data_merged

In [16]:
data["label"].value_counts()

label
fumarola      92387
no_visible    92122
erupcion      32626
inactivo      23257
Name: count, dtype: int64

In [18]:
#TODO: find sequential pictures, remove edges of sequence prop. to length

In [19]:
data.to_feather("../popocatepetl-data-v3/data_map.feather")

In [20]:

from datasets import Dataset, Image as dsImage, Value as scalar, ClassLabel, Features, DatasetInfo
from PIL import Image
import torchvision

In [21]:
data["image"]=data["path"]
data.drop(columns=["path"],inplace=True)

In [22]:
class_names=data["label"].unique().tolist()
class_names

['fumarola', 'erupcion', 'inactivo', 'no_visible']

In [27]:
freq=data["label"].value_counts()
freq

label
fumarola      92387
no_visible    92122
erupcion      32626
inactivo      23257
Name: count, dtype: int64

In [28]:
mx=np.max(freq)/freq
mx=mx/mx.sum()
mx

label
fumarola      0.113546
no_visible    0.113872
erupcion      0.321528
inactivo      0.451054
Name: count, dtype: float64

In [29]:
print(mx[class_names].tolist())

[0.11354586666416572, 0.321527676806911, 0.45105396153855953, 0.11387249499036363]


In [30]:

features=Features({
    "label":ClassLabel(num_classes=len(class_names), names=class_names),
    "image":dsImage(),
    "video":scalar("string"),
    "id":scalar("int32"),
})

In [31]:
info=DatasetInfo(
description = "A dataset on Popocatépetl, Mexico's most notorious volcano.",
homepage = "https://github.com/mirluvams",
#license = "MIT",
features = features,
version = "3.0.0",
)

In [35]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 240392 entries, 0 to 224094
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   label   240392 non-null  object
 1   id      240392 non-null  int32 
 2   video   240392 non-null  object
 3   image   240392 non-null  object
dtypes: int32(1), object(3)
memory usage: 8.3+ MB


In [37]:
dataset=Dataset.from_pandas(data, features=features, preserve_index=False)

In [38]:
dataset=dataset.train_test_split(test_size=.1, seed=36918, stratify_by_column="label")

In [39]:
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'image', 'video', 'id'],
        num_rows: 216352
    })
    test: Dataset({
        features: ['label', 'image', 'video', 'id'],
        num_rows: 24040
    })
})

In [40]:
dataset.save_to_disk("../dataset")

Saving the dataset (0/11 shards):   0%|          | 0/216352 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/24040 [00:00<?, ? examples/s]

In [41]:
dataset.push_to_hub("mirluvams/popocatepetl-v3", private=True)

Uploading the dataset shards:   0%|          | 0/11 [00:00<?, ?it/s]

Map:   0%|          | 0/19669 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/197 [00:00<?, ?ba/s]

Map:   0%|          | 0/19669 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/197 [00:00<?, ?ba/s]

...

Creating parquet from Arrow format:   0%|          | 0/121 [00:00<?, ?ba/s]

Map:   0%|          | 0/12020 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/121 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/630 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/mirluvams/popocatepetl-v3/commit/bab5f0bb0ac0d17a32a434573323a6185093d84f', commit_message='Upload dataset', commit_description='', oid='bab5f0bb0ac0d17a32a434573323a6185093d84f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/mirluvams/popocatepetl-v3', endpoint='https://huggingface.co', repo_type='dataset', repo_id='mirluvams/popocatepetl-v3'), pr_revision=None, pr_num=None)